# Doc2Vec - Train từ đầu (From Scratch)

## Giới thiệu

Trong notebook này, chúng ta sẽ xây dựng và huấn luyện mô hình Doc2Vec **hoàn toàn từ đầu** trên dữ liệu 20 Newsgroups mà không sử dụng bất kỳ pretrained model nào.

### Doc2Vec là gì?

Doc2Vec (Document to Vector) là phương pháp học biểu diễn vector cho toàn bộ tài liệu/văn bản dài, mở rộng từ Word2Vec. 

### Hai kiến trúc Doc2Vec:

- **PV-DM (Distributed Memory)**: Sử dụng document vector kết hợp với word vectors để dự đoán từ tiếp theo, giữ thứ tự từ.
- **PV-DBOW (Distributed Bag of Words)**: Chỉ sử dụng document vector để dự đoán các từ trong tài liệu, bỏ qua thứ tự từ.

In [ ]:
# Cài đặt các thư viện cần thiết# pip install gensim scikit-learn nltk matplotlib seaborn

In [ ]:
# Import các thư việnimport gensimfrom gensim.models.doc2vec import Doc2Vec, TaggedDocumentimport nltknltk.download('punkt', quiet=True)nltk.download('punkt_tab', quiet=True)from sklearn.datasets import fetch_20newsgroupsfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrixfrom sklearn.manifold import TSNEimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport timeimport randomimport warningswarnings.filterwarnings('ignore')# Thiết lập seedSEED = 42random.seed(SEED)np.random.seed(SEED)

## Load dữ liệu 20 Newsgroups

In [ ]:
# Định nghĩa 4 categoriescategories = ['sci.med', 'sci.space', 'rec.sport.baseball', 'talk.politics.misc']# Load dữ liệu train và testtrain_data = fetch_20newsgroups(    subset='train',    categories=categories,    remove=('headers', 'footers', 'quotes'),    random_state=SEED)test_data = fetch_20newsgroups(    subset='test',    categories=categories,    remove=('headers', 'footers', 'quotes'),    random_state=SEED)# In thông tin datasetprint(f"Số lượng mẫu train: {len(train_data.data)}")print(f"Số lượng mẫu test: {len(test_data.data)}")print(f"Categories: {train_data.target_names}")# Ví dụ văn bảnprint("\n" + "="*50)print("Ví dụ văn bản mẫu:")print("="*50)print(train_data.data[0][:300])

## Tiền xử lý văn bản

In [ ]:
# Tiền xử lýdef preprocess_text(text):    """    Tokenize và lowercase    """    tokens = nltk.word_tokenize(text.lower())    tokens = [word for word in tokens if word.isalpha()]    return tokens# Tạo TaggedDocumenttagged_documents = []for i, doc in enumerate(train_data.data):    tokens = preprocess_text(doc)    tagged_documents.append(TaggedDocument(words=tokens, tags=[i]))print(f"Số TaggedDocuments: {len(tagged_documents)}")print(f"Ví dụ: tags={tagged_documents[0].tags}, {len(tagged_documents[0].words)} tokens")

## Train mô hình PV-DM (Distributed Memory)

In [ ]:
# Train PV-DM từ đầuprint("="*60)print("TRAIN PV-DM TỪ ĐẦU")print("="*60)print("Bắt đầu training...")start_time = time.time()# Khởi tạo modelmodel_dm = Doc2Vec(    vector_size=150,    window=5,    min_count=2,    workers=4,    epochs=40,    dm=1,    seed=SEED)# Build vocab và trainmodel_dm.build_vocab(tagged_documents)model_dm.train(    tagged_documents,    total_examples=model_dm.corpus_count,    epochs=model_dm.epochs)# Thời giandm_time = time.time() - start_timeprint(f"Hoàn thành trong {dm_time:.2f} giây")# Lưu modelmodel_dm.save("doc2vec_pvdm_scratch.model")print("Đã lưu: doc2vec_pvdm_scratch.model")

## Train mô hình PV-DBOW (Distributed Bag of Words)

In [ ]:
# Train PV-DBOW từ đầuprint("="*60)print("TRAIN PV-DBOW TỪ ĐẦU")print("="*60)print("Bắt đầu training...")start_time = time.time()model_dbow = Doc2Vec(    vector_size=150,    window=5,    min_count=2,    workers=4,    epochs=40,    dm=0,    seed=SEED)model_dbow.build_vocab(tagged_documents)model_dbow.train(    tagged_documents,    total_examples=model_dbow.corpus_count,    epochs=model_dbow.epochs)dbow_time = time.time() - start_timeprint(f"Hoàn thành trong {dbow_time:.2f} giây")model_dbow.save("doc2vec_pvdbow_scratch.model")print("Đã lưu: doc2vec_pvdbow_scratch.model")

## Trích xuất vector

In [ ]:
# Trích xuất vector cho trainprint("Trích xuất vector train...")train_vectors_dm = np.array([model_dm.dv[i] for i in range(len(train_data.data))])train_vectors_dbow = np.array([model_dbow.dv[i] for i in range(len(train_data.data))])# Trích xuất vector cho testprint("Trích xuất vector test...")test_vectors_dm = []test_vectors_dbow = []for doc in test_data.data:    tokens = preprocess_text(doc)    test_vectors_dm.append(model_dm.infer_vector(tokens, epochs=40))    test_vectors_dbow.append(model_dbow.infer_vector(tokens, epochs=40))test_vectors_dm = np.array(test_vectors_dm)test_vectors_dbow = np.array(test_vectors_dbow)print(f"PV-DM train: {train_vectors_dm.shape}, test: {test_vectors_dm.shape}")print(f"PV-DBOW train: {train_vectors_dbow.shape}, test: {test_vectors_dbow.shape}")

## Phân loại với Logistic Regression

In [ ]:
# Lấy nhãny_train = train_data.targety_test = test_data.targetresults = {}# PV-DMprint("="*60)PHÂN LOẠI - PV-DMprint("="*60)clf_dm = LogisticRegression(max_iter=1000, random_state=SEED)clf_dm.fit(train_vectors_dm, y_train)y_pred_dm = clf_dm.predict(test_vectors_dm)acc_dm = accuracy_score(y_test, y_pred_dm)results['PV-DM'] = acc_dmprint(f"Accuracy: {acc_dm:.4f}")print(classification_report(y_test, y_pred_dm, target_names=categories))# PV-DBOWprint("="*60)PHÂN LOẠI - PV-DBOWprint("="*60)clf_dbow = LogisticRegression(max_iter=1000, random_state=SEED)clf_dbow.fit(train_vectors_dbow, y_train)y_pred_dbow = clf_dbow.predict(test_vectors_dbow)acc_dbow = accuracy_score(y_test, y_pred_dbow)results['PV-DBOW'] = acc_dbowprint(f"Accuracy: {acc_dbow:.4f}")print(classification_report(y_test, y_pred_dbow, target_names=categories))

## So sánh kết quả

In [ ]:
# Bảng so sánhprint("\n" + "="*60)SO SÁNH KẾT QUẢ - TRAIN TỪ ĐẦUprint("="*60)print(f"{'Model':<20} {'Accuracy':>15}")print("-"*35)for model, acc in results.items():    print(f"{model:<20} {acc:>15.4f}")best_model = max(results.items(), key=lambda x: x[1])print(f"\nTốt nhất: {best_model[0]} - Accuracy: {best_model[1]:.4f}")

In [ ]:
# Biểu đồ so sánhplt.figure(figsize=(8, 5))models = list(results.keys())accs = list(results.values())colors = ['#1f77b4', '#ff7f0e']bars = plt.bar(models, accs, color=colors)plt.ylabel('Accuracy')plt.title('So sánh - Train từ đầu', fontsize=14, fontweight='bold')plt.ylim(0, 1)for bar, acc in zip(bars, accs):    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,            f'{acc:.4f}', ha='center')plt.tight_layout()plt.show()

## Confusion Matrix

In [ ]:
# Confusion matrix cho model tốt nhấtbest_key = max(results.items(), key=lambda x: x[1])[0]if best_key == 'PV-DM':    y_pred_best = y_pred_dmelse:    y_pred_best = y_pred_dbowcm = confusion_matrix(y_test, y_pred_best)plt.figure(figsize=(10, 8))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',            xticklabels=categories, yticklabels=categories)plt.title(f'Confusion Matrix - {best_key}', fontsize=14, fontweight='bold')plt.xlabel('Predicted')plt.ylabel('True')plt.xticks(rotation=45, ha='right')plt.tight_layout()plt.show()

## Demo tìm tài liệu tương tự

In [ ]:
# Tìm tài liệu tương tựtest_idx = 50test_doc = test_data.data[test_idx]print(f"Tài liệu test (index {test_idx}):")print("="*50)print(test_doc[:200])# Trích xuất vector và tìm similartest_tokens = preprocess_text(test_doc)test_vec = model_dm.infer_vector(test_tokens, epochs=40)similar = model_dm.dv.most_similar(positive=[test_vec], topn=3)print("\n3 tài liệu train gần nhất:")print("="*50)for doc_id, sim in similar:    print(f"Doc {doc_id}, Similarity: {sim:.4f}")    print(train_data.data[doc_id][:150] + "...\n")

## t-SNE Visualization

In [ ]:
# t-SNEn_per_class = 50selected_indices = []for i in range(len(categories)):    idx = np.where(y_test == i)[0][:n_per_class]    selected_indices.extend(idx)selected_indices = np.array(selected_indices)selected_vectors = test_vectors_dm[selected_indices]selected_labels = y_test[selected_indices]print("Áp dụng t-SNE...")tsne = TSNE(n_components=2, random_state=42, perplexity=30)vectors_2d = tsne.fit_transform(selected_vectors)plt.figure(figsize=(10, 7))colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']for i, cat in enumerate(categories):    idx = selected_labels == i    plt.scatter(vectors_2d[idx, 0], vectors_2d[idx, 1],               c=colors[i], label=cat, alpha=0.7, s=50)plt.title('t-SNE - Train từ đầu (PV-DM)', fontsize=14, fontweight='bold')plt.xlabel('t-SNE 1')plt.ylabel('t-SNE 2')plt.legend()plt.tight_layout()plt.show()

## Phân tích kết quả

### Tóm tắt kết quả train từ đầu:

- PV-DM: accuracy sẽ được hiển thị sau khi chạy
- PV-DBOW: accuracy sẽ được hiển thị sau khi chạy

### Nhận xét:

1. **PV-DBOW** thường nhanh hơn và cho accuracy cao hơn trên dataset này
2. **PV-DM** giữ thứ tự từ nên có thể tốt hơn cho một số bài toán ngữ nghĩa
3. Training từ đầu cho phép tùy chỉnh theo domain cụ thể

### Hạn chế:
- Dataset 20 Newsgroups relatively nhỏ
- Model được train riêng cho task này, không thể reuse cho task khác
- Không tận dụng được knowledge từ large corpora